# Import

In [1]:
import pandas as pd
import numpy as np

# Constantes

In [2]:
# Processed Data Path
PATH_PROCESSED = "../data/processed/"

# Palpites
FILE_TIPS = "palpites_fg_processados.csv"
# Paises
FILE_PAISES = "apoio_paises.csv"

# ETL

## Leitura e Join com apoio

In [3]:
df_paises = pd.read_csv(PATH_PROCESSED + FILE_PAISES)

In [4]:
df_tips_table = pd.read_csv(PATH_PROCESSED + FILE_TIPS)
df_tips_table["nm_pais"] = df_tips_table["nm_time_casa"]

In [5]:
df_tips_table_2 = pd.merge(df_tips_table, df_paises, on='nm_pais', how='left')
df_tips_table_2.drop(["nm_pais"], axis=1, inplace=True)

In [6]:
df_tips_table_3 = df_tips_table_2.dropna()
df_tips_table_3['id_pais'] = df_tips_table_3['id_pais'].astype(int)

C:\Users\ferol\AppData\Local\Temp\ipykernel_31720\4274215766.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tips_table_3['id_pais'] = df_tips_table_3['id_pais'].astype(int)


## Gerar classificação

In [7]:
df = df_tips_table_3.copy()

# resultados por mandante
home = df.rename(columns={
    "nm_time_casa": "team",
    "nm_time_fora": "opp",
    "vl_time_casa": "gf",
    "vl_time_fora": "ga",
})

In [8]:
home["pts"] = np.select(
    [home["gf"] > home["ga"], home["gf"] == home["ga"]],
    [3, 1],
    default=0,
)

home["v"] = (home["gf"] > home["ga"]).astype(int)
home["e"] = (home["gf"] == home["ga"]).astype(int)
home["d"] = (home["gf"] < home["ga"]).astype(int)

In [9]:
home["v"] = (home["gf"] > home["ga"]).astype(int)
home["e"] = (home["gf"] == home["ga"]).astype(int)
home["d"] = (home["gf"] < home["ga"]).astype(int)

In [10]:
# resultados por visitante (espelha o jogo)
away = df.rename(columns={
    "nm_time_fora": "team",
    "nm_time_casa": "opp",
    "vl_time_fora": "gf",
    "vl_time_casa": "ga",
})

In [11]:
away["pts"] = np.select(
    [away["gf"] > away["ga"], away["gf"] == away["ga"]],
    [3, 1],
    default=0,
)

away["v"] = (away["gf"] > away["ga"]).astype(int)
away["e"] = (away["gf"] == away["ga"]).astype(int)
away["d"] = (away["gf"] < away["ga"]).astype(int)

In [23]:
# concatena e agrega
team_rows = pd.concat([home, away], ignore_index=True)

In [13]:
base_table = (
    team_rows
    .groupby(["nm_player", "nm_grpo", "team"], as_index=False)
    .agg(
        pts=("pts", "sum"),
        jogos=("team", "size"),
        v=("v", "sum"),
        e=("e", "sum"),
        d=("d", "sum"),
        gp=("gf", "sum"),
        gc=("ga", "sum"),
    )
)

base_table["sg"] = base_table["gp"] - base_table["gc"]

base_table

,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg
0,ana nath,A,Coreia do Sul,1,3,0,1,2,3,7,-4
1,ana nath,A,Europa D,7,3,2,1,0,7,2,5
2,ana nath,A,México,6,3,2,0,1,9,5,4
3,ana nath,A,África do Sul,3,3,1,0,2,4,9,-5
4,ana nath,B,Canadá,7,3,2,1,0,7,3,4
...,...,...,...,...,...,...,...,...,...,...,...
187,washington,K,Uzbequistão,7,3,2,1,0,7,2,5
188,washington,L,Croácia,3,3,1,0,2,4,5,-1
189,washington,L,Gana,5,3,1,2,0,7,4,3
190,washington,L,Inglaterra,1,3,0,1,2,1,5,-4


In [54]:
base_table['rk'] = base_table.groupby(['nm_player','nm_grpo'])['pts'].rank(method='max', ascending=False).astype(int)


base_table[base_table['nm_player'] == 'ana nath'].sort_values(['nm_grpo','rk'], ascending=[True, True])

,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg,rk
1,ana nath,A,Europa D,7,3,2,1,0,7,2,5,1
2,ana nath,A,México,6,3,2,0,1,9,5,4,2
3,ana nath,A,África do Sul,3,3,1,0,2,4,9,-5,3
0,ana nath,A,Coreia do Sul,1,3,0,1,2,3,7,-4,4
4,ana nath,B,Canadá,7,3,2,1,0,7,3,4,1
7,ana nath,B,Suíça,4,3,1,1,1,5,6,-1,2
5,ana nath,B,Catar,3,3,1,0,2,4,6,-2,4
6,ana nath,B,Europa A,3,3,1,0,2,4,5,-1,4
9,ana nath,C,Escócia,7,3,2,1,0,6,2,4,1
10,ana nath,C,Haiti,4,3,1,1,1,7,8,-1,2


## Critérios de Desempate

A classificação de seleções em cada grupo será determinada pelos pontos obtidos em todas as partidas do grupos. Se duas ou mais seleções empatarem em pontos, os critérios a seguir serão usados para determinar a classificação:

a. Mais pontos obtidos na partida de grupo jogada entre as seleções em questão;\
b. Maior saldo de gols na partida de grupo jogada entre as seleções em questão;\
c. Mais gols marcados na partida de grupo jogada entre as seleções em questão;

Se, depois de aplicados os critérios de A a C, seleções ainda estiverem empatadas, estes critérios serão aplicados novamente exclusivamente às partidas entre as seleções que ainda estão empatadas para determinar sua classificação final. Se este procedimento não levar a uma decisão, os critérios de D a H se aplicam.

d. Maior saldo de gols em todas as partidas do grupo;\
e. Mais gols marcados em todas as partidas do grupo;\
f. Melhor conduta ("fair play") em todas as partidas do grupo (apenas uma dedução pode ser aplicada a um jogador ou a um membro da comissão técnica/dirigente por partida):
- Cartão amarelo: −1 ponto;
- Cartão vermelho indireto (segundo cartão amarelo): −3 pontos;
- Cartão vermelho direto: −4 pontos;
- Cartão amarelo e cartão vermelho direto: −5 pontos;

g. Melhor posição no Ranking Mundial da FIFA mais recente;\
h. Melhor posição em Rankings Mundiais da FIFA mais antigos progressivamente até que as seleções sejam separadas;

In [ ]:
# pontos por confronto direto
h2h = (
    team_rows
    .groupby(["nm_player", "nm_grpo", "team", "opp"], as_index=False)["pts"]
    .sum()
)

h2h[(h2h['nm_player'] == 'ana nath') & (h2h['nm_grpo'] == 'B')].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,opp,pts
12,ana nath,B,Canadá,Catar,3
13,ana nath,B,Canadá,Europa A,3
14,ana nath,B,Canadá,Suíça,1
15,ana nath,B,Catar,Canadá,0
16,ana nath,B,Catar,Europa A,0
17,ana nath,B,Catar,Suíça,3
18,ana nath,B,Europa A,Canadá,0
19,ana nath,B,Europa A,Catar,3
20,ana nath,B,Europa A,Suíça,0
21,ana nath,B,Suíça,Canadá,1


In [52]:
tot = base_table[["nm_player", "nm_grpo", "team", "pts"]].rename(columns={"pts": "total_pts"})

tot[(tot['nm_player'] == 'ana nath') & (tot['nm_grpo'] == 'B')].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,total_pts
4,ana nath,B,Canadá,7
5,ana nath,B,Catar,3
6,ana nath,B,Europa A,3
7,ana nath,B,Suíça,4


In [58]:
h2h_2 = h2h.merge(tot, on=["nm_player", "nm_grpo", "team"])

h2h_2[(h2h_2['nm_player'] == 'ana nath') & (h2h_2['nm_grpo'] == 'B')].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,opp,pts,total_pts
12,ana nath,B,Canadá,Catar,3,7
13,ana nath,B,Canadá,Europa A,3,7
14,ana nath,B,Canadá,Suíça,1,7
15,ana nath,B,Catar,Canadá,0,3
16,ana nath,B,Catar,Europa A,0,3
17,ana nath,B,Catar,Suíça,3,3
18,ana nath,B,Europa A,Canadá,0,3
19,ana nath,B,Europa A,Catar,3,3
20,ana nath,B,Europa A,Suíça,0,3
21,ana nath,B,Suíça,Canadá,1,4


In [62]:
h2h_3 = h2h_2.merge(
    tot.rename(columns={"team": "opp", "total_pts": "opp_total_pts"}),
    on=["nm_player", "nm_grpo", "opp"]
)

h2h_3[(h2h_3['nm_player'] == 'ana nath') & (h2h_3['nm_grpo'] == 'B')].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,opp,pts,total_pts,opp_total_pts
12,ana nath,B,Canadá,Catar,3,7,3
13,ana nath,B,Canadá,Europa A,3,7,3
14,ana nath,B,Canadá,Suíça,1,7,4
15,ana nath,B,Catar,Canadá,0,3,7
16,ana nath,B,Catar,Europa A,0,3,3
17,ana nath,B,Catar,Suíça,3,3,4
18,ana nath,B,Europa A,Canadá,0,3,7
19,ana nath,B,Europa A,Catar,3,3,3
20,ana nath,B,Europa A,Suíça,0,3,4
21,ana nath,B,Suíça,Canadá,1,4,7


In [66]:
# mantém só jogos entre equipes com o mesmo total de pontos
h2h_tied = h2h_3[h2h_3["total_pts"] == h2h_3["opp_total_pts"]]

# h2h_tied[(h2h_tied['nm_player'] == 'ana nath') & (h2h_tied['nm_grpo'] == 'B')].sort_values(['nm_grpo'], ascending=[True])

h2h_tied[h2h_tied['nm_player'] == 'ana nath'].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,opp,pts,total_pts,opp_total_pts
16,ana nath,B,Catar,Europa A,0,3,3
19,ana nath,B,Europa A,Catar,3,3,3
37,ana nath,D,Austrália,Paraguai,3,3,3
42,ana nath,D,Paraguai,Austrália,0,3,3
48,ana nath,E,Costa do Marfim,Equador,3,3,3
52,ana nath,E,Equador,Costa do Marfim,0,3,3
77,ana nath,G,Nova Zelândia,Irã,1,5,5
75,ana nath,G,Nova Zelândia,Bélgica,1,5,5
74,ana nath,G,Irã,Nova Zelândia,1,5,5
72,ana nath,G,Irã,Bélgica,1,5,5


In [67]:
h2h_pts = (
    h2h_tied
    .groupby(["nm_player", "nm_grpo", "team"], as_index=False)["pts"]
    .sum()
    .rename(columns={"pts": "h2h_pts"})
)

h2h_pts[(h2h_pts['nm_player'] == 'ana nath') & (h2h_pts['nm_grpo'] == 'B')].sort_values(['nm_grpo'], ascending=[True])


,nm_player,nm_grpo,team,h2h_pts
0,ana nath,B,Catar,0
1,ana nath,B,Europa A,3
